# Offline Pretrained Model Training (Target-Specific)

This notebook trains and exports pretrained supervised models for inference-only pipeline usage.

Training design:
- Regression: use only `data/Smart_Manufacturing_Maintenance_Dataset/smart_maintenance_dataset.csv` with target `Failure_Prob`.
- Classification: use only `data/Smart_Manufacturing_Maintenance_Dataset/smart_maintenance_dataset.csv` with target `Maintenance_Priority`.
- For each eligible target column, split data into 80% train and 20% hold-out test.
- Train candidate models, pick the best per target, and save one bundle per target.

Evaluation design:
- New inference cells load saved bundles and run prediction on the held-out 20% split.
- Report target-wise metrics (Regression: R2, MSE, RMSE, MAE; Classification: Accuracy, Precision, Recall, F1).

In [1]:
from pathlib import Path
import json
import re
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    precision_recall_fscore_support,
    f1_score,
    precision_score,
    recall_score,
    )

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.svm import SVR, SVC

In [2]:
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'data').exists():
    PROJECT_DIR = PROJECT_DIR.parent

ARTIFACT_DIR = PROJECT_DIR / 'artifacts' / 'pretrained_models'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

REGRESSION_DATA_PATH = PROJECT_DIR / 'data' / 'Smart_Manufacturing_Maintenance_Dataset' / 'smart_maintenance_dataset.csv'
CLASSIFICATION_DATA_PATH = PROJECT_DIR / 'data' / 'Smart_Manufacturing_Maintenance_Dataset' / 'smart_maintenance_dataset.csv'

print('Project dir:', PROJECT_DIR)
print('Artifacts dir:', ARTIFACT_DIR)
print('Regression dataset:', REGRESSION_DATA_PATH)
print('Classification dataset:', CLASSIFICATION_DATA_PATH)

Project dir: /Users/rakshith/Desktop/Smart-manufacturing-mas/smart_manufacturing_mas
Artifacts dir: /Users/rakshith/Desktop/Smart-manufacturing-mas/smart_manufacturing_mas/artifacts/pretrained_models
Regression dataset: /Users/rakshith/Desktop/Smart-manufacturing-mas/smart_manufacturing_mas/data/Smart_Manufacturing_Maintenance_Dataset/smart_maintenance_dataset.csv
Classification dataset: /Users/rakshith/Desktop/Smart-manufacturing-mas/smart_manufacturing_mas/data/Smart_Manufacturing_Maintenance_Dataset/smart_maintenance_dataset.csv


In [3]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])

    transformers = []
    if numeric_cols:
        transformers.append(('num', num_pipe, numeric_cols))
    if categorical_cols:
        transformers.append(('cat', cat_pipe, categorical_cols))

    if not transformers:
        raise ValueError("No numeric or categorical columns found in X")

    pre = ColumnTransformer(
        transformers=transformers,
        remainder='drop'
    )
    return pre


def sanitize_name(name: str) -> str:
    return re.sub(r'[^A-Za-z0-9_]+', '_', str(name)).strip('_')


def save_bundle(bundle: dict, bundle_file: str) -> None:
    out_path = ARTIFACT_DIR / bundle_file
    joblib.dump(bundle, out_path)


def evaluate_regression(y_true, y_pred) -> dict:
    mse = float(mean_squared_error(y_true, y_pred))
    return {
        'r2': float(r2_score(y_true, y_pred)),
        'mse': mse,
        'rmse': float(np.sqrt(mse)),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }


def evaluate_classification(y_true, y_pred) -> dict:
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'classification_report': classification_report(y_true, y_pred),
    }


def get_regression_targets(df: pd.DataFrame, excluded_patterns: list[str] = None) -> list[str]:
    """
    Find numeric columns that could be regression targets.
    Exclude columns that match sensor/measurement naming patterns by default.
    """
    if excluded_patterns is None:
        excluded_patterns = ['_C', '_mm_s', '_Bar', '_dB', '_min', '_USD', '_pct', '_Prob']
    
    targets = []
    for col in df.columns:
        if col.lower().endswith('_id') or col.lower() == 'id':
            continue
        
        # Skip sensor/measurement columns
        if any(pattern in col for pattern in excluded_patterns):
            continue
            
        if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) > 20:
            targets.append(col)
    return targets


def get_classification_targets(df: pd.DataFrame, min_cardinality: int = 2, max_cardinality: int = 50) -> list[str]:
    """
    Find columns that could be classification targets.
    Integer columns with 2-50 unique values, or object/category/bool columns with 2-50 unique values.
    """
    targets = []
    for col in df.columns:
        if col.lower().endswith('_id') or col.lower() == 'id':
            continue
        
        s = df[col]
        nunique = s.nunique(dropna=True)
        
        # Skip if too many or too few classes
        if nunique < min_cardinality or nunique > max_cardinality:
            continue
        
        if pd.api.types.is_bool_dtype(s) or str(s.dtype) in ('object', 'category'):
            # For object dtype, skip if mostly NaN
            if s.isna().sum() / len(s) > 0.8:
                continue
            targets.append(col)
        elif pd.api.types.is_integer_dtype(s):
            targets.append(col)
    
    return targets


def train_best_model_for_target(df: pd.DataFrame, target: str, problem_type: str, model_zoo: dict):
    X = df.drop(columns=[target]).copy()
    y = df[target].copy()

    # Remove obvious identifier columns from features
    id_like = [c for c in X.columns if c.lower().endswith('_id') or c.lower() == 'id']
    if id_like:
        X = X.drop(columns=id_like)

    if problem_type == 'classification':
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

    leaderboard = []
    bundles = []

    for model_name, estimator in model_zoo.items():
        try:
            pre = build_preprocessor(X_train)
            pipe = Pipeline([('preprocessor', pre), ('model', estimator)])
            pipe.fit(X_train, y_train)
            preds = pipe.predict(X_test)

            if problem_type == 'classification':
                metrics = evaluate_classification(y_test, preds)
                score = metrics['f1_weighted']
            else:
                metrics = evaluate_regression(y_test, preds)
                score = metrics['r2']

            leaderboard.append({
                'target': target,
                'model_name': model_name,
                **{k: v for k, v in metrics.items() if k != 'classification_report'}
            })

            bundle_payload = {
                'model_name': model_name,
                'problem_type': problem_type,
                'target_column': target,
                'feature_columns': X_train.columns.tolist(),
                'pipeline': pipe,
                'metrics': metrics,
            }
            if problem_type == 'regression':
                train_preds = pipe.predict(X_train)
                bundle_payload['train_prediction_stats'] = {
                    'mean': float(np.mean(train_preds)),
                    'std': float(np.std(train_preds)),
                }

            bundles.append((score, bundle_payload, X_test, y_test))
        except Exception as e:
            print(f"  ⚠️  Model {model_name} failed for target '{target}': {str(e)[:100]}")
            continue

    if not bundles:
        raise ValueError(f"No models trained successfully for target '{target}'")

    reverse = True
    bundles = sorted(bundles, key=lambda x: x[0], reverse=reverse)
    best_score, best_bundle, best_X_test, best_y_test = bundles[0]

    best_model_name = best_bundle['model_name']
    bundle_file = f"{problem_type}__{sanitize_name(target)}__{best_model_name}.joblib"
    save_bundle(best_bundle, bundle_file)

    entry = {
        'model_name': best_model_name,
        'bundle_file': bundle_file,
        'target_column': target,
        'feature_columns': best_bundle['feature_columns'],
        'metrics': best_bundle['metrics'],
    }

    return entry, pd.DataFrame(leaderboard), best_X_test, best_y_test


In [4]:
reg_df = pd.read_csv(REGRESSION_DATA_PATH)
reg_df = reg_df.drop(columns=['Machine_ID', 'Maintenance_Priority'], errors='ignore')

# For regression dataset, explicitly use Failure_Prob as target
# and remove the classification label from the feature set.
REGRESSION_TARGET = 'Failure_Prob'

if REGRESSION_TARGET not in reg_df.columns:
    raise ValueError(f"Target '{REGRESSION_TARGET}' not found. Columns: {reg_df.columns.tolist()}")

print(f"Training regression model for target: {REGRESSION_TARGET}")

regression_models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(random_state=42),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'SVR': SVR(kernel='rbf', C=1.0),
}

regression_entries = []
regression_leaderboards = []
regression_holdout_sets = {}

entry, lb_df, X_test, y_test = train_best_model_for_target(
    df=reg_df,
    target=REGRESSION_TARGET,
    problem_type='regression',
    model_zoo=regression_models,
)
regression_entries.append(entry)
regression_leaderboards.append(lb_df)
regression_holdout_sets[REGRESSION_TARGET] = {
    'X_test': X_test,
    'y_test': y_test,
    'bundle_file': entry['bundle_file'],
}

regression_summary = pd.DataFrame([
    {
        'target': e['target_column'],
        'best_model': e['model_name'],
        'r2': e['metrics'].get('r2'),
        'mse': e['metrics'].get('mse'),
        'rmse': e['metrics'].get('rmse'),
        'mae': e['metrics'].get('mae'),
    }
    for e in regression_entries
])

regression_summary.sort_values('r2', ascending=False)


Training regression model for target: Failure_Prob


,target,best_model,r2,mse,rmse,mae
0,Failure_Prob,LinearRegression,0.997078,0.000008,0.002906,0.002505


In [5]:
cls_df = pd.read_csv(CLASSIFICATION_DATA_PATH)
cls_df = cls_df.drop(columns=['Machine_ID', 'Failure_Prob'], errors='ignore')

# For classification dataset, explicitly use Maintenance_Priority as target
# (auto-detection picks up sensor measurements as regression targets)
CLASSIFICATION_TARGET = 'Maintenance_Priority'

if CLASSIFICATION_TARGET not in cls_df.columns:
    raise ValueError(f"Target '{CLASSIFICATION_TARGET}' not found. Columns: {cls_df.columns.tolist()}")

print(f"Training classification model for target: {CLASSIFICATION_TARGET}")
print(f"  Target values: {sorted(cls_df[CLASSIFICATION_TARGET].unique())}")

classification_models = {
    'LogisticRegression': LogisticRegression(max_iter=2000, random_state=42),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'SVC': SVC(kernel='rbf', C=1.0, probability=False),
}

classification_entries = []
classification_leaderboards = []
classification_holdout_sets = {}

try:
    entry, lb_df, X_test, y_test = train_best_model_for_target(
        df=cls_df,
        target=CLASSIFICATION_TARGET,
        problem_type='classification',
        model_zoo=classification_models,
    )
    classification_entries.append(entry)
    classification_leaderboards.append(lb_df)
    classification_holdout_sets[CLASSIFICATION_TARGET] = {
        'X_test': X_test,
        'y_test': y_test,
        'bundle_file': entry['bundle_file'],
    }
    print(f"✓ Successfully trained classification model for {CLASSIFICATION_TARGET}")
except Exception as e:
    print(f"✗ Failed to train classification model: {e}")
    raise

classification_summary = pd.DataFrame([
    {
        'target': e['target_column'],
        'best_model': e['model_name'],
        'accuracy': e['metrics'].get('accuracy'),
        'precision_macro': e['metrics'].get('precision_macro'),
        'recall_macro': e['metrics'].get('recall_macro'),
        'f1_macro': e['metrics'].get('f1_macro'),
        'f1_weighted': e['metrics'].get('f1_weighted'),
    }
    for e in classification_entries
])

classification_summary.sort_values('f1_weighted', ascending=False)


Training classification model for target: Maintenance_Priority
  Target values: [np.int64(1), np.int64(2), np.int64(3)]
✓ Successfully trained classification model for Maintenance_Priority


,target,best_model,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,Maintenance_Priority,LogisticRegression,0.98951,0.991534,0.953961,0.971137,0.989287


In [6]:
registry = {
    'regression': regression_entries,
    'classification': classification_entries,
}

registry_path = ARTIFACT_DIR / 'registry.json'
with registry_path.open('w', encoding='utf-8') as f:
    json.dump(registry, f, indent=2)

print('Saved registry:', registry_path)
print('Saved regression target-model bundles:', len(regression_entries))
print('Saved classification target-model bundles:', len(classification_entries))

Saved registry: /Users/rakshith/Desktop/Smart-manufacturing-mas/smart_manufacturing_mas/artifacts/pretrained_models/registry.json
Saved regression target-model bundles: 1
Saved classification target-model bundles: 1


In [7]:
# Inference on held-out 20% split for regression bundles
reg_inference_rows = []

for target, payload in regression_holdout_sets.items():
    bundle = joblib.load(ARTIFACT_DIR / payload['bundle_file'])
    y_pred = bundle['pipeline'].predict(payload['X_test'])
    m = evaluate_regression(payload['y_test'], y_pred)
    reg_inference_rows.append({
        'target': target,
        'bundle': payload['bundle_file'],
        'r2': m['r2'],
        'mse': m['mse'],
        'rmse': m['rmse'],
        'mae': m['mae'],
    })

pd.DataFrame(reg_inference_rows).sort_values('r2', ascending=False)

,target,bundle,r2,mse,rmse,mae
0,Failure_Prob,regression__Failure_Prob__LinearRegression.joblib,0.997078,0.000008,0.002906,0.002505


In [8]:
# Inference on held-out 20% split for classification bundles
cls_inference_rows = []

for target, payload in classification_holdout_sets.items():
    bundle = joblib.load(ARTIFACT_DIR / payload['bundle_file'])
    y_pred = bundle['pipeline'].predict(payload['X_test'])
    m = evaluate_classification(payload['y_test'], y_pred)
    cls_inference_rows.append({
        'target': target,
        'bundle': payload['bundle_file'],
        'accuracy': m['accuracy'],
        'precision_macro': m['precision_macro'],
        'recall_macro': m['recall_macro'],
        'f1_macro': m['f1_macro'],
        'f1_weighted': m['f1_weighted'],
    })

pd.DataFrame(cls_inference_rows).sort_values('f1_weighted', ascending=False)

,target,bundle,accuracy,precision_macro,recall_macro,f1_macro,f1_weighted
0,Maintenance_Priority,classification__Maintenance_Priority__Logistic...,0.98951,0.991534,0.953961,0.971137,0.989287
